<a href="https://colab.research.google.com/github/RexCrowSS1/HeartMLDecisionTree/blob/main/HeartMLDecisionTree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow scikit-learn numpy

In [ ]:
# ===========================================
# CELL 1: Mount Google Drive & Load Dataset
# ===========================================
from google.colab import drive
import pandas as pd
import numpy as np

# 1. Hubungkan Colab ke Google Drive
drive.mount('/content/drive')

# 2. Load dataset
file_path = '/content/drive/MyDrive/DataSets/heart_cleveland_upload.csv'
df = pd.read_csv(file_path)

print("\n--- Dataset Berhasil Dimuat! ---")
print(f"Jumlah baris: {df.shape[0]}, Jumlah kolom: {df.shape[1]}")
display(df.head())

Mounted at /content/drive

--- Dataset Berhasil Dimuat! ---
Jumlah baris: 297, Jumlah kolom: 14


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,condition
0,69,1,0,160,234,1,2,131,0,0.1,1,1,0,0
1,69,0,0,140,239,0,0,151,0,1.8,0,2,0,0
2,66,0,0,150,226,0,0,114,0,2.6,2,0,0,0
3,65,1,0,138,282,1,2,174,0,1.4,1,1,0,1
4,64,1,0,110,211,0,2,144,1,1.8,1,0,0,0


In [ ]:
# ===========================================
# CELL 2: Preprocessing & Standardisasi
# ===========================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Pisahkan Fitur (X) dan Target/Label (y)
# CATATAN: Pastikan nama kolom targetmu benar. Biasanya di dataset Cleveland namanya 'condition' atau 'target'.
# Ganti 'condition' di bawah ini jika nama kolom targetmu berbeda.
nama_kolom_target = 'condition'

X = df.drop(columns=[nama_kolom_target]) # Mengambil semua kolom KECUALI target
y = df[nama_kolom_target] # Hanya mengambil kolom target (biasanya isinya 0 atau 1)

# 2. Bagi data menjadi Training (80%) dan Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Standardisasi Fitur (SANGAT PENTING UNTUK SVM)
print("Sedang melakukan standardisasi fitur...")
scaler = StandardScaler()

# Model belajar standar dari data training, lalu mengubahnya
X_train_scaled = scaler.fit_transform(X_train)
# Data testing hanya diubah mengikuti standar dari data training
X_test_scaled = scaler.transform(X_test)

print("Standardisasi selesai! Data siap dimasukkan ke model SVM.")

Sedang melakukan standardisasi fitur...
Standardisasi selesai! Data siap dimasukkan ke model SVM.


In [ ]:
# ===========================================
# CELL 3: OPTIMALISASI DECISION TREE DENGAN GRIDSEARCHCV (GREEDY SEARCH)
# ===========================================
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

# 1. Tentukan variasi parameter Decision Tree untuk dicoba via Grid Search
param_grid = {
    'criterion': ['gini', 'entropy'],        # Metrik untuk greedy splitting
    'max_depth': [None, 3, 5, 7, 10],        # Membatasi kedalaman pohon (mencegah overfitting)
    'min_samples_split': [2, 5, 10],         # Jumlah sampel minimal untuk membuat cabang baru
    'min_samples_leaf': [1, 2, 4]            # Jumlah sampel minimal yang harus ada di daun akhir
}

print("Sedang mencari kombinasi parameter terbaik untuk Decision Tree (Grid Search)...")
# cv=5 artinya kita menggunakan 5-Fold Cross Validation agar hasilnya akurat
# class_weight='balanced' membantu menangani ketidakseimbangan data jika ada
grid = GridSearchCV(DecisionTreeClassifier(class_weight='balanced', random_state=42), param_grid, refit=True, cv=5, verbose=1)

# 2. Latih proses pencarian dengan data training yang sudah di-scaled
grid.fit(X_train_scaled, y_train)

# 3. Tampilkan parameter terbaik yang ditemukan komputer
print("\n--- PROSES SELESAI ---")
print(f"Kombinasi Parameter Terbaik: {grid.best_params_}")

# 4. Prediksi menggunakan model yang sudah dioptimalkan
y_pred_opt = grid.predict(X_test_scaled)

# ==========================================
# HASIL EVALUASI BARU
# ==========================================
print("\n--- HASIL EVALUASI SETELAH OPTIMALISASI DECISION TREE ---")
print(f"Accuracy Score Baru: {accuracy_score(y_test, y_pred_opt) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_opt))

Sedang mencari kombinasi parameter terbaik untuk Decision Tree (Grid Search)...
Fitting 5 folds for each of 90 candidates, totalling 450 fits

--- PROSES SELESAI ---
Kombinasi Parameter Terbaik: {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 2, 'min_samples_split': 10}

--- HASIL EVALUASI SETELAH OPTIMALISASI DECISION TREE ---
Accuracy Score Baru: 66.67%

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.66      0.68        32
           1       0.63      0.68      0.66        28

    accuracy                           0.67        60
   macro avg       0.67      0.67      0.67        60
weighted avg       0.67      0.67      0.67        60

